In [1]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("extended_attention_rag_testing.pdf")
docs = loader.load()
docs

[Document(metadata={'source': 'extended_attention_rag_testing.pdf', 'page': 0}, page_content='Extended Knowledge Document for RAG\n Testing\n Attention, Transformers, Embeddings, RAG and Sample Company Facts\n1. Introduction to Artificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems capable of\nperforming tasks that normally require human intelligence. These tasks include language\nunderstanding, image recognition, reasoning, planning, prediction, and decision support. AI systems\ncan be rule-based or data-driven, although modern AI increasingly relies on machine learning.\nMachine learning is a branch of AI in which algorithms learn patterns from examples. Supervised\nlearning uses labeled data, unsupervised learning discovers patterns without predefined labels, and\nreinforcement learning trains an agent using rewards and penalties. Deep learning is a subfield of\nmachine learning based on neural networks containing multi

In [2]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
text_splitter.split_documents(docs)[:5]

[Document(metadata={'source': 'extended_attention_rag_testing.pdf', 'page': 0}, page_content='Extended Knowledge Document for RAG\n Testing\n Attention, Transformers, Embeddings, RAG and Sample Company Facts\n1. Introduction to Artificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems capable of\nperforming tasks that normally require human intelligence. These tasks include language\nunderstanding, image recognition, reasoning, planning, prediction, and decision support. AI systems\ncan be rule-based or data-driven, although modern AI increasingly relies on machine learning.\nMachine learning is a branch of AI in which algorithms learn patterns from examples. Supervised\nlearning uses labeled data, unsupervised learning discovers patterns without predefined labels, and\nreinforcement learning trains an agent using rewards and penalties. Deep learning is a subfield of\nmachine learning based on neural networks containing multi

In [3]:
documents=text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'extended_attention_rag_testing.pdf', 'page': 0}, page_content='Extended Knowledge Document for RAG\n Testing\n Attention, Transformers, Embeddings, RAG and Sample Company Facts\n1. Introduction to Artificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems capable of\nperforming tasks that normally require human intelligence. These tasks include language\nunderstanding, image recognition, reasoning, planning, prediction, and decision support. AI systems\ncan be rule-based or data-driven, although modern AI increasingly relies on machine learning.\nMachine learning is a branch of AI in which algorithms learn patterns from examples. Supervised\nlearning uses labeled data, unsupervised learning discovers patterns without predefined labels, and\nreinforcement learning trains an agent using rewards and penalties. Deep learning is a subfield of\nmachine learning based on neural networks containing multi

In [4]:
from langchain_ollama import OllamaEmbeddings
from langchain_classic.vectorstores import FAISS
embeddings=OllamaEmbeddings(model="nomic-embed-text")

db=FAISS.from_documents(documents[:30], embeddings)

In [5]:
db

In [6]:
query="An attention function can be described as mapping a query "
result=db.similarity_search(query)
result[0].page_content

'4. Attention Mechanism\nThe attention mechanism helps a neural network focus on the most relevant portions of its input.\nRather than assigning equal importance to every token, attention computes weights that indicate\nwhich tokens should receive more focus while producing a representation.\nAttention is particularly useful for language because the meaning of one word may depend on another\nword located far away in the sentence. The mechanism provides a direct way to model these\nrelationships.\n5. Query, Key, and Value\nA common attention formulation uses Query, Key, and Value vectors. A Query represents what the\nmodel is looking for. A Key describes information that can be matched against a Query. A Value\ncontains the information associated with a Key.\nThe model compares a Query with Keys to calculate relevance scores. These scores are usually\nnormalized with a softmax operation. The normalized scores become attention weights, which'

In [7]:
from langchain_ollama import ChatOllama
## Load Ollama LAMA2 LLM model
llm=ChatOllama(model="llama2")
llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, model='llama2')

In [8]:
## Design ChatPrompt Template
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context.
Think step by step before providing a detailed answer.
I will tip you $1000 if the user finds the answer helpful.
<context>
{context}
</context>
Question: {input}""")

In [9]:
## Chain Introduction
## Create Stuff Docment Chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain=create_stuff_documents_chain(llm,prompt)

In [10]:
"""
Retrievers: A retriever is an interface that returns documents given
 an unstructured query. It is more general than a vector store.
 A retriever does not need to be able to store documents, only to
 return (or retrieve) them. Vector stores can be used as the backbone
 of a retriever, but there are other types of retrievers as well.
 https://python.langchain.com/docs/modules/data_connection/retrievers/
"""

retriever=db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000019130B85E20>, search_kwargs={})

In [11]:
"""
Retrieval chain:This chain takes in a user inquiry, which is then
passed to the retriever to fetch relevant documents. Those documents
(and original inputs) are then passed to an LLM to generate a response
https://python.langchain.com/docs/modules/chains/
"""
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [12]:
response=retrieval_chain.invoke({"input":"Scaled Dot-Product Attention"})

In [13]:
response['answer']

'In the context provided, Scaled Dot-Product Attention is a key component of the Transformer architecture, which is a neural network architecture introduced in the 2017 paper "Attention Is All You Need". The Scaled Dot-Product Attention mechanism is used to compute attention weights, which are used to weight the importance of different tokens in a sequence.\n\nThe Scaled Dot-Product Attention mechanism consists of three components: Query, Key, and Value. The Query represents what the model is looking for, the Key describes information that can be matched against the Query, and the Value contains the information associated with the Key. The model compares the Query with the Key to calculate relevance scores, which are then normalized with a softmax operation to produce attention weights.\n\nThe attention weights are used to compute a weighted sum of the Value, where the weights are the attention scores. This allows the model to focus on the most relevant parts of the input sequence when